In [1]:
import pandas as pd
import numpy as np

In [4]:
# Load EM-DAT data
data = pd.read_excel("emdat.xlsx")

In [7]:
# 1. Filter to Western Hemisphere (Americas)
region_col = "Continent" if "Continent" in data.columns else "Region"

americas = [
    "Americas",
    "North America",
    "Northern America",
    "Central America",
    "South America",
    "Caribbean"
]

data = data[
    data[region_col].isin(americas) &
    (data["Disaster Group"] == "Natural") &
    (data["Start Year"].between(2000, 2025))
].copy()

# 2. Convert deaths and affected to numeric
for col in ["Total Deaths", "Total Affected"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")


In [8]:
# 3. Extract unique disaster IDs
# Remove country suffix so one disaster affecting
# multiple countries is counted only once.
data["Event ID"] = (
    data["DisNo."].astype(str)
    .str.replace(r"-[A-Z]{3}$", "", regex=True)
)

# Combine country-level impacts for each event
events = data.groupby(
    ["Event ID", "Disaster Type"], as_index=False
).agg(
    deaths=("Total Deaths", lambda x: x.sum(min_count=1)),
    affected=("Total Affected", lambda x: x.sum(min_count=1))
)

In [9]:
# 4. Calculate frequency and average severity indicators
summary = events.groupby("Disaster Type").agg(
    event_count=("Event ID", "nunique"),
    avg_deaths=("deaths", "mean"),
    avg_affected=("affected", "mean"),
    deaths_reported=("deaths", "count"),
    affected_reported=("affected", "count")
).reset_index()

# Percentage of all disasters
total_events = events["Event ID"].nunique()

summary["frequency_pct"] = (
    summary["event_count"] / total_events * 100
)

In [10]:
# 5. Normalize averages to a 0-100 scale
# Log scaling reduces the influence of extreme disasters
def normalize_log(series):
    maximum = series.max()
    if pd.isna(maximum) or maximum <= 0:
        return series * 0

    return np.log1p(series) / np.log1p(maximum) * 100

summary["death_score"] = normalize_log(summary["avg_deaths"])
summary["affected_score"] = normalize_log(summary["avg_affected"])

In [13]:
# 6. Calculate weighted severity
summary["severity_score"] = (
    0.70 * summary["death_score"] +
    0.30 * summary["affected_score"]
)

In [14]:
# 7. Round results and sort by frequency
summary = summary.sort_values(
    "frequency_pct", ascending=False
).round(2)

print(summary.to_string(index=False))

      Disaster Type  event_count  avg_deaths  avg_affected  deaths_reported  affected_reported  frequency_pct  death_score  affected_score  severity_score
              Flood          838       27.06      98143.05              591                783          41.10        41.12           81.64           53.28
              Storm          614       34.84     358019.59              509                441          30.11        44.14           90.83           58.15
           Wildfire          146       11.69      98463.37               72                123           7.16        31.34           81.66           46.44
         Earthquake          101     3320.09     153923.57               69                 99           4.95       100.00           84.84           95.45
Mass movement (wet)           95       38.25       7802.64               89                 73           4.66        45.26           63.66           50.78
            Drought           82       72.50    1301672.63            